# Fraud-score label-provenance diagnostic

This read-only diagnostic measures how `fraud_score` aligns with `is_fraud` in the fixed validation period. It is **not** a training run or evidence that `fraud_score` is available before authorization. The exact alignment may indicate label leakage. The final-test period is excluded from the query.

If the score producer confirms that this field is generated independently and is available before the transaction decision, it may be evaluated as a pre-decision input in a separately versioned model. Until then, do not use it to claim real-time model performance.

In [ ]:
from pyspark.sql import functions as F

SOURCE_TABLE = 'workspace.bank_silver.transactions'
DELTA_VERSION = 1
VALIDATION_START = '2025-07-01 00:00:00'
VALIDATION_END = '2026-01-01 00:00:00'

source = spark.read.option('versionAsOf', DELTA_VERSION).table(SOURCE_TABLE)
validation = source.where(
    (F.col('transaction_date') >= F.to_timestamp(F.lit(VALIDATION_START)))
    & (F.col('transaction_date') < F.to_timestamp(F.lit(VALIDATION_END)))
)
score_band = (
    F.when(F.col('fraud_score').isNull(), 'missing')
    .when(F.col('fraud_score') >= 70, '>=70')
    .when(F.col('fraud_score') >= 50, '50-69')
    .when(F.col('fraud_score') >= 30, '30-49')
    .otherwise('<30')
)
bands = (
    validation.select(score_band.alias('score_band'), 'is_fraud')
    .groupBy('score_band')
    .agg(
        F.count('*').alias('transactions'),
        F.sum(F.when(F.col('is_fraud'), 1).otherwise(0)).alias('fraud'),
        F.sum(F.when(~F.col('is_fraud'), 1).otherwise(0)).alias('non_fraud'),
    )
    .orderBy('score_band')
)
display(bands)

# Only five aggregate rows are collected; no transaction or customer records are exported.
band_counts = {r['score_band']: r.asDict() for r in bands.collect()}
for threshold, included_bands in {
    30: ('30-49', '50-69', '>=70'),
    50: ('50-69', '>=70'),
    70: ('>=70',),
}.items():
    rows = sum(int(band_counts[b]['transactions']) for b in included_bands)
    fraud = sum(int(band_counts[b]['fraud']) for b in included_bands)
    print({'threshold': threshold, 'alerts': rows, 'fraud': fraud, 'precision': fraud / rows if rows else None})

## Interpretation

Compare the observed label association with the score's documented generation time. A threshold's observed precision is not a deployable metric if the score was calculated after fraud detection, derived from the label, or unavailable at authorization. Even when timely, confirm the result on a later untouched temporal period before operational use.